# Việc 9: fine-tune với augment nhẹ (kiểu close-mosaic)

Lấy `last.keras` đã train xong, train tiếp **15 epoch, lr 2e-4**, augment nhẹ (`--aug-light`: zoom 0,85–1,15, màu nhẹ,
không nhiễu / mờ / giảm phân giải), không ảnh rỗng giả (`--empty-p 0`). Kiến trúc giữ nguyên (198.196 tham số, 4 toán tử).

1. **Kiểm chứng** trên `v7_s4_p3` (5.400 ảnh train, đo trên 600 ảnh val): so mAP val trước và sau fine-tune.
2. **Chỉ khi bước 1 tăng ≥ +0,0015** mới fine-tune bản G (6.000 ảnh) theo đúng công thức đó. Không đạt thì tự bỏ qua.

**Trước khi chạy:** Add Input dữ liệu cuộc thi + Dataset chứa `edgeai_viec9.zip`; Settings: **GPU T4 x2**, **Internet On**.
Chạy bằng **Save Version → Save & Run All (Commit)**. Khoảng 30 phút (chỉ bước 1) hoặc 1 giờ (cả 2 bước).
Kết quả: `viec9.zip` ở tab **Output** → tải về Downloads, báo Claude.

## 1. Cài Keras 3.15.1 và kiểm GPU

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "keras==3.15.1"], check=True)
chk = ("import os; os.environ['KERAS_BACKEND']='torch'; import keras, torch; "
       "print('keras', keras.__version__, '| torch', torch.__version__); "
       "print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHONG THAY GPU'); "
       "assert keras.__version__ == '3.15.1' and torch.cuda.is_available()")
subprocess.run([sys.executable, "-c", chk], check=True)

## 2. Tìm dữ liệu và code

In [ ]:
import json, os, shutil, time, zipfile
from pathlib import Path

INP, WORK = Path("/kaggle/input"), Path("/kaggle/working")


def find(test, what, base=INP, depth=5):
    for k in range(1, depth + 1):
        for d in sorted(base.glob("/".join(["*"] * k))):
            if test(d):
                return d
    raise FileNotFoundError(f"Không thấy {what} trong {base}: kiểm tra đã Add Input chưa")


DATA = find(lambda d: (d / "train" / "images").is_dir() and (d / "train" / "labels").is_dir(), "dữ liệu cuộc thi")
try:
    PKG = find(lambda d: (d / "pipeline" / "det" / "train_det.py").is_file()
               and "aug-light" in (d / "pipeline" / "det" / "train_det.py").read_text(encoding="utf-8"), "gói edgeai_viec9")
except FileNotFoundError:
    z = find(lambda d: d.is_file() and d.name == "edgeai_viec9.zip", "edgeai_viec9.zip")
    PKG = WORK / "pkg"
    zipfile.ZipFile(z).extractall(PKG)
PIPE = WORK / "pipeline"
shutil.copytree(PKG / "pipeline", PIPE, dirs_exist_ok=True)
W_V7, W_G = PKG / "weights" / "v7_s4_p3_last.keras", PKG / "weights" / "g_s4_p3_last.keras"
assert W_V7.is_file() and W_G.is_file(), "thiếu file trọng số trong gói"
n_img = len(list((DATA / "train" / "images").glob("*.png")))
assert n_img == 6000, f"{n_img} ảnh train, cần 6000"
print("DATA:", DATA, f"({n_img} ảnh) | PKG:", PKG, "| commit", (PIPE / "GIT_COMMIT").read_text().strip())

ENV = {**os.environ, "KERAS_BACKEND": "torch", "EDGE_IMGSZ": "320x240", "EDGE_STRIDES": "4,8,16",
       "PYTHONUNBUFFERED": "1", "PYTHONIOENCODING": "utf-8", "CUDA_VISIBLE_DEVICES": "0"}
FT = ["--epochs", "15", "--batch", "16", "--lr", "2e-4", "--eval-every", "3", "--empty-p", "0",
      "--aug-light", "--arch", "64,64,72", "--no-export"]


def train(name, init, val_frac):
    out = WORK / "runs" / name
    cmd = [sys.executable, str(PIPE / "det/train_det.py"), "--data", str(DATA), "--out", str(out),
           "--val-frac", str(val_frac), "--init-weights", str(init), *FT]
    print(">", " ".join(cmd), flush=True)
    t0, log = time.time(), []
    p = subprocess.Popen(cmd, cwd=WORK, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        log.append(line)
        print(line, end="", flush=True)
    code = p.wait()
    print(f"\nXong sau {(time.time() - t0) / 60:.1f} phút, mã thoát {code}")
    assert code == 0, "Lệnh lỗi, xem log phía trên"
    text = "".join(log)
    assert "stride: 4,8,16" in text and "198,196" in text, "Sai cấu hình (không thấy stride 4,8,16 / 198,196 tham số)"
    return out

## 3. Bước 1: kiểm chứng trên `v7_s4_p3` (khoảng 30 phút)

In [ ]:
OUT1 = train("v9_ft_v7", W_V7, 0.1)
ref = json.load(open(PKG / "ref" / "split_v7_s4_p3.json"))["val_ids"]
assert json.load(open(OUT1 / "split.json"))["val_ids"] == ref, "Tập val KHÁC v7_s4_p3 -> phép so sánh không hợp lệ"
meta = json.load(open(OUT1 / "train_meta.json"))
logs = json.load(open(OUT1 / "log.json"))
m0, m_last, m_best = meta["val_map50_init"], logs[-1]["map50"], meta["val_map50_float"]
GAIN = m_last - m0
print(f"Tập val trùng 100% với v7_s4_p3. mAP50 val (float): trước {m0:.4f} -> epoch cuối {m_last:.4f} "
      f"(tốt nhất {m_best:.4f}) | tăng {GAIN:+.4f}")
RUN_G = GAIN >= 0.0015
print(">>> ĐẠT ngưỡng +0,0015: chạy bước 2 cho bản G" if RUN_G else ">>> KHÔNG đạt ngưỡng: bỏ qua bước 2, giữ bản G gốc")

## 4. Bước 2: fine-tune bản G (chỉ chạy khi bước 1 đạt, khoảng 30 phút)

In [ ]:
OUT2 = None
if RUN_G:
    OUT2 = train("v9_ft_g", W_G, 0)
    assert json.load(open(OUT2 / "split.json"))["val_ids"] == [], "Tập val phải rỗng (--val-frac 0)"
else:
    print("Bỏ qua bước 2.")

## 5. Đóng gói kết quả

In [ ]:
Z = WORK / "viec9.zip"
with zipfile.ZipFile(Z, "w", zipfile.ZIP_STORED) as z:
    for out in (OUT1, OUT2):
        if out is None:
            continue
        for f in ("last.keras", "best.keras", "log.json", "split.json", "train_meta.json"):
            if (out / f).exists():
                z.write(out / f, f"{out.name}/{f}")
for r in logs:
    print(r["epoch"], round(r["map50"], 4))
for d in (PIPE, WORK / "pkg", WORK / "runs"):
    shutil.rmtree(d, ignore_errors=True)
print(f"Đã tạo {Z} ({Z.stat().st_size / 2**20:.1f} MB)", "| có bản G" if OUT2 else "| chỉ có bước 1")